In [3]:
!pip install torch torchvision torchaudio

   ---------------------------------------- 0.0/122.1 MB ? eta -:--:--
   ---------------------------------------- 0.5/122.1 MB 4.4 MB/s eta 0:00:28
    --------------------------------------- 2.9/122.1 MB 9.6 MB/s eta 0:00:13
   -- ------------------------------------- 8.4/122.1 MB 18.6 MB/s eta 0:00:07
   --- ------------------------------------ 10.7/122.1 MB 17.4 MB/s eta 0:00:07
   ---- ----------------------------------- 14.4/122.1 MB 16.6 MB/s eta 0:00:07
   ----- ---------------------------------- 17.3/122.1 MB 15.8 MB/s eta 0:00:07
   ------ --------------------------------- 20.2/122.1 MB 15.4 MB/s eta 0:00:07
   ------- -------------------------------- 22.3/122.1 MB 14.8 MB/s eta 0:00:07
   ------- -------------------------------- 23.3/122.1 MB 14.1 MB/s eta 0:00:08
   ------- -------------------------------- 24.1/122.1 MB 12.6 MB/s eta 0:00:08
   -------- ------------------------------- 24.6/122.1 MB 11.8 MB/s eta 0:00:09
   -------- ------------------------------- 25.2/122.1

In [4]:
import torch
print(torch.__version__)

2.13.0+cpu


In [5]:
import math
#import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# Scaled Dot-Product Attention
# ============================================================
def scaled_dot_product_attention(q, k, v, mask=None):
    """
    q, k, v : (batch, heads, seq_len, d_k)
    mask    : (1,1,L,L) or (batch,1,L,L)
    """

    d_k = q.size(-1)

    scores = torch.matmul(q, k.transpose(-2, -1))
    scores = scores / math.sqrt(d_k)

    if mask is not None:
        mask = mask.to(torch.bool)
        scores = scores.masked_fill(~mask, torch.finfo(scores.dtype).min)

    attention = F.softmax(scores, dim=-1)

    output = torch.matmul(attention, v)

    return output, attention


# ============================================================
# Multi-Head Attention
# ============================================================
class MultiHeadAttention(nn.Module):

    def __init__(self, d_model, num_heads):
        super().__init__()

        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)

        self.w_o = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        batch_size, seq_len, _ = x.size()

        x = x.view(batch_size, seq_len, self.num_heads, self.d_k)

        return x.transpose(1, 2)

    def combine_heads(self, x):
        batch_size, _, seq_len, _ = x.size()

        x = x.transpose(1, 2).contiguous()

        return x.view(batch_size, seq_len, self.d_model)

    def forward(self, q, k, v, mask=None):

        q = self.split_heads(self.w_q(q))
        k = self.split_heads(self.w_k(k))
        v = self.split_heads(self.w_v(v))

        output, attention = scaled_dot_product_attention(
            q, k, v, mask
        )

        output = self.combine_heads(output)

        output = self.w_o(output)

        return output, attention


# ============================================================
# Positional Encoding
# ============================================================
class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(max_len).unsqueeze(1).float()

        div_term = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)

        self.register_buffer("pe", pe)

    def forward(self, x):

        seq_len = x.size(1)

        return x + self.pe[:, :seq_len].to(x.device)


# ============================================================
# Feed Forward Network
# ============================================================
class FeedForward(nn.Module):

    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        return self.net(x)


# ============================================================
# Transformer Encoder Block
# ============================================================
class TransformerEncoderBlock(nn.Module):

    def __init__(self,
                 d_model,
                 num_heads,
                 d_ff,
                 dropout=0.1):

        super().__init__()

        self.attention = MultiHeadAttention(
            d_model,
            num_heads
        )

        self.ffn = FeedForward(
            d_model,
            d_ff,
            dropout
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):

        y = self.norm1(x)

        attn_out, _ = self.attention(
            y,
            y,
            y,
            mask
        )

        x = x + self.dropout(attn_out)

        y = self.norm2(x)

        ff_out = self.ffn(y)

        x = x + self.dropout(ff_out)

        return x


# ============================================================
# Tiny Transformer Encoder
# ============================================================
class TinyTransformerEncoder(nn.Module):

    def __init__(self,
                 vocab_size,
                 d_model=64,
                 num_heads=4,
                 d_ff=256,
                 num_layers=2,
                 max_len=100):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        self.position = PositionalEncoding(
            d_model,
            max_len
        )

        self.layers = nn.ModuleList([
            TransformerEncoderBlock(
                d_model,
                num_heads,
                d_ff
            )
            for _ in range(num_layers)
        ])

        self.norm = nn.LayerNorm(d_model)

    def forward(self, token_ids, mask=None):

        x = self.embedding(token_ids)

        x = x * math.sqrt(self.embedding.embedding_dim)

        x = self.position(x)

        for layer in self.layers:
            x = layer(x, mask)

        x = self.norm(x)

        return x


# ============================================================
# Causal Mask
# ============================================================
def causal_mask(seq_len, device=None):

    mask = torch.tril(
        torch.ones(
            seq_len,
            seq_len,
            dtype=torch.bool,
            device=device,
        )
    )

    return mask.unsqueeze(0).unsqueeze(0)


# ============================================================
# Demo
# ============================================================
if __name__ == "__main__":

    torch.manual_seed(0)

    vocab_size = 1000
    batch_size = 2
    seq_len = 8

    model = TinyTransformerEncoder(
        vocab_size=vocab_size,
        d_model=64,
        num_heads=4,
        d_ff=256,
        num_layers=2,
        max_len=100,
    )

    dummy_tokens = torch.randint(
        0,
        vocab_size,
        (batch_size, seq_len)
    )

    # -------------------------
    # Bidirectional
    # -------------------------
    out = model(dummy_tokens)

    print("Bidirectional output:", out.shape)

    # -------------------------
    # Causal
    # -------------------------
    mask = causal_mask(
        seq_len,
        dummy_tokens.device
    )

    out = model(
        dummy_tokens,
        mask
    )

    print("Causal output:", out.shape)



Bidirectional output: torch.Size([2, 8, 64])
Causal output: torch.Size([2, 8, 64])
